In [ ]:
import time
import json
import os
import requests

#Datasets A - Anime and Manga: IDs, description, genres, etc
DATA_DIR = os.path.join("..", "data")  
ANIME_DATA_PATH = os.path.join(DATA_DIR, "anime_data.jsonl")
MANGA_DATA_PATH = os.path.join(DATA_DIR, "manga_data.jsonl")

ANILIST_URL = "https://graphql.anilist.co" #Getting data from AniList API

QUERY = """
query ($page: Int, $type: MediaType) {
  Page(page: $page, perPage: 50) {
    pageInfo { hasNextPage }
    media(type: $type, sort: POPULARITY_DESC) {
      id
      title { romaji english }
      genres
      tags { name }
      description
      coverImage { large }
      averageScore
      popularity
      format
    }
  }
}
"""

def fetch_page(page, media_type="ANIME"):
    variables = {"page": page, "type": media_type}
    response = requests.post(ANILIST_URL, json={"query": QUERY, "variables": variables})
    response.raise_for_status()
    return response.json()["data"]["Page"]
i=1
data = []
#Get Anime data
while True:
    x=fetch_page(i)
    data.extend(x['media'])
    
    if i == 100: #API caps at 5000 requests 
        with open(ANIME_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        break
    i+=1
    
    if i % 10 == 0:
        with open(ANIME_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        data.clear()
        
    time.sleep(2) #Ensures we wont have too many requests too quick

data.clear()
i=1
#Get Manga data
while True:
    x=fetch_page(i, media_type="MANGA")
    data.extend(x['media'])
    
    if i == 100: 
        with open(MANGA_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        break
    i+=1
    
    if i % 10 == 0:
        with open(MANGA_DATA_PATH, 'a') as file:
            for item in data:
                file.write(f'{json.dumps(item)}\n')
        data.clear()
        
    time.sleep(2) 



In [1]:
import kagglehub
import os
import json
import numpy as np
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))


print(animes.shape)
display(animes.head())

# Genres
genres = animes[['animeID', 'genres']].copy() 
display(genres)

# Ratings
ratings_array = np.load(os.path.join(path, "ratings.npy"))
ratings_df = pd.DataFrame(ratings_array, columns=["user_id", "anime_id", "rating"])

display(ratings_df[:25])



(20237, 12)


,animeID,title,alternative_title,type,year,score,episodes,mal_url,sequel,image_url,genres,genres_detailed
0,1,Howl's Moving Castle,Howl no Ugoku Shiro,MOVIE,2004,8.41,1,https://myanimelist.net/anime/431,False,https://cdn.myanimelist.net/images/anime/1470/...,"['Adventure', 'Award Winning', 'Drama', 'Fanta...","['action', 'adventure', 'age gap', 'air force'..."
1,2,Death Note,NaN,TV,2006,8.63,37,https://myanimelist.net/anime/1535,False,https://cdn.myanimelist.net/images/anime/1079/...,"['Supernatural', 'Suspense']","['achronological order', 'acting', 'adapted in..."
2,3,Problem Children Are Coming from Another World...,Mondaiji-tachi ga Isekai kara Kuru Sou desu yo?,TV,2013,7.42,10,https://myanimelist.net/anime/15315,False,https://cdn.myanimelist.net/images/anime/12/43...,"['Action', 'Comedy', 'Fantasy']","['action', 'alternative world', 'anthropomorph..."
3,4,BTOOOM!,Btooom!,TV,2012,7.34,12,https://myanimelist.net/anime/14345,False,https://cdn.myanimelist.net/images/anime/4/409...,"['Action', 'Sci-Fi', 'Suspense']","['achronological order', 'action', 'adventure'..."
4,5,Sword Art Online,NaN,TV,2012,7.5,25,https://myanimelist.net/anime/11757,False,https://cdn.myanimelist.net/images/anime/11/39...,"['Action', 'Adventure', 'Fantasy', 'Romance']","['action', 'action drama', 'adventure', 'alter..."


,animeID,genres
0,1,"['Adventure', 'Award Winning', 'Drama', 'Fanta..."
1,2,"['Supernatural', 'Suspense']"
2,3,"['Action', 'Comedy', 'Fantasy']"
3,4,"['Action', 'Sci-Fi', 'Suspense']"
4,5,"['Action', 'Adventure', 'Fantasy', 'Romance']"
...,...,...
20232,20233,['Fantasy']
20233,20234,['Fantasy']
20234,20235,['Fantasy']
20235,20236,[]


,user_id,anime_id,rating
0,1,1,10
1,1,2,10
2,1,3,7
3,1,4,10
4,1,5,10
5,1,6,10
6,1,7,10
7,1,8,10
8,1,9,6
9,1,10,10


In [7]:
print(f"Rating Distrubition: {ratings_df['rating'].value_counts().sort_index()}\n")
print(f"Ratings per user: {ratings_df.groupby('user_id').size().describe()}\n")
print(f"Ratings per anime: {ratings_df.groupby('anime_id').size().describe()}\n")

Rating Distrubition: rating
0       186337
1      1643276
2      1273086
3      2008890
4      4750391
5      7471134
6     16391767
7     30726885
8     35294326
9     21767496
10    26656908
Name: count, dtype: int64

Ratings per user: count    1.774522e+06
mean     8.349882e+01
std      1.560422e+02
min      5.000000e+00
25%      1.000000e+01
50%      2.900000e+01
75%      9.200000e+01
max      1.088100e+04
dtype: float64

Ratings per anime: count     20237.000000
mean       7321.761921
std       32883.434898
min           1.000000
25%          49.000000
50%         264.000000
75%        2127.000000
max      956713.000000
dtype: float64



In [8]:
ratings_clean = ratings_df[ratings_df['rating'] != 0].copy()
ratings_clean['is_positive'] = (ratings_clean['rating'] >= 8).astype(int)

print(ratings_clean['is_positive'].value_counts(normalize=True))

is_positive
1    0.565728
0    0.434272
Name: proportion, dtype: float64
